# Parameter-Efficient Fine-Tuning with LoRA

## Introduction

Fine-tuning large language models on downstream tasks is a powerful technique, but it comes with significant challenges:

1. **Memory constraints**: Full fine-tuning requires storing gradients, optimizer states, and activations for all parameters (billions of them)
2. **Catastrophic forgetting**: Updating all weights can destroy the general knowledge the model learned during pretraining
3. **Storage costs**: Maintaining separate copies of multi-billion parameter models for each task is impractical

**Parameter-Efficient Fine-Tuning (PEFT)** solves these problems by updating only a small subset of parameters while keeping the pretrained model frozen.

In this notebook, we'll explore:
- **LoRA (Low-Rank Adaptation)**: The most popular PEFT method
- **QLoRA**: Quantized LoRA for even greater efficiency
- **Other PEFT methods**: Prefix tuning, prompt tuning, and adapters
- **Practical implementation**: From scratch and with HuggingFace PEFT

## Setup

Let's import the libraries we'll need and configure our environment.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
from typing import Optional, List, Tuple
from dataclasses import dataclass
import math

from aiml_notebooks import get_device, set_seed, count_parameters

%load_ext autoreload
%autoreload 2

Set random seed for reproducibility and configure the device.

In [ ]:
set_seed(42)
device = get_device(prefer_cpu=True)  # LoRA works well on CPU for demos
print(f"Using device: {device}")

## 1. The Fine-Tuning Problem

### 1.1 Full Fine-Tuning: Why It Fails at Scale

Let's quantify the memory requirements for full fine-tuning. We'll use a simple transformer as an example.

In [ ]:
# Example: GPT-2 Small has 124M parameters
# GPT-3 has 175B parameters
# LLaMA-2 70B has 70B parameters

def calculate_training_memory(num_params_millions, precision_bytes=4):
    """
    Calculate approximate memory requirements for full fine-tuning.
    
    Memory components:
    - Model weights: num_params * precision_bytes
    - Gradients: num_params * precision_bytes (same size as model)
    - Optimizer states: num_params * 8 (Adam keeps 2 moment estimates per param)
    - Activations: ~num_params * 4 (rough estimate)
    """
    num_params = num_params_millions * 1e6
    
    model_memory = num_params * precision_bytes
    gradients_memory = num_params * precision_bytes
    optimizer_memory = num_params * 8  # Adam: 2 states * 4 bytes each
    activations_memory = num_params * 4
    
    total_memory = model_memory + gradients_memory + optimizer_memory + activations_memory
    
    return {
        'model_gb': model_memory / 1e9,
        'gradients_gb': gradients_memory / 1e9,
        'optimizer_gb': optimizer_memory / 1e9,
        'activations_gb': activations_memory / 1e9,
        'total_gb': total_memory / 1e9
    }

# Calculate for different model sizes
models = {
    'GPT-2 Small': 124,
    'GPT-2 Large': 774,
    'GPT-3': 175000,
    'LLaMA-2 7B': 7000,
    'LLaMA-2 70B': 70000
}

print("Full Fine-Tuning Memory Requirements (FP32):")
print("=" * 70)
for name, size in models.items():
    mem = calculate_training_memory(size)
    print(f"\n{name} ({size}M params):")
    print(f"  Model: {mem['model_gb']:.1f} GB")
    print(f"  Gradients: {mem['gradients_gb']:.1f} GB")
    print(f"  Optimizer: {mem['optimizer_gb']:.1f} GB")
    print(f"  Activations: {mem['activations_gb']:.1f} GB")
    print(f"  TOTAL: {mem['total_gb']:.1f} GB")

**Key insight**: Fine-tuning LLaMA-2 70B requires ~1,260 GB of memory - impossible on consumer GPUs!

Even GPT-2 Small (124M params) needs ~2.2 GB just for training overhead. This is where PEFT methods shine.

### 1.2 The Solution: Update Only a Subset

The key insight: **we don't need to update all parameters**. The model already knows a lot - we just need to adapt it slightly.

PEFT methods achieve this by:
1. **Freezing** the pretrained model weights
2. **Adding** a small number of trainable parameters
3. **Training** only these new parameters

This can reduce trainable parameters from billions to millions (0.1-1% of original).

## 2. LoRA Fundamentals

### 2.1 The Core Intuition

**Low-Rank Adaptation (LoRA)** is based on a key observation: during fine-tuning, weight updates have **low intrinsic rank**.

What does this mean?
- A weight matrix $W \in \mathbb{R}^{d \times k}$ might have full rank $\min(d, k)$
- But the **change** $\Delta W$ during fine-tuning can be approximated by a low-rank matrix
- Low-rank means: most information is in just a few dimensions

**Example**: A 1000×1000 matrix has 1M parameters, but a rank-8 approximation has only 16K parameters (1.6%)!

In [ ]:
# Demonstrate low-rank decomposition
d, k = 1000, 1000
rank = 8

# Full rank matrix
full_params = d * k

# Low-rank decomposition: W ≈ B @ A where B is d×r and A is r×k
low_rank_params = d * rank + rank * k

reduction = (1 - low_rank_params / full_params) * 100

print(f"Full rank matrix: {d}×{k} = {full_params:,} parameters")
print(f"Low-rank (r={rank}): ({d}×{rank}) + ({rank}×{k}) = {low_rank_params:,} parameters")
print(f"\nParameter reduction: {reduction:.1f}%")
print(f"Compression ratio: {full_params / low_rank_params:.1f}x")

### 2.2 The LoRA Equation

LoRA modifies a pretrained weight matrix $W_0 \in \mathbb{R}^{d \times k}$ by adding a low-rank update:

$$W = W_0 + \Delta W = W_0 + BA$$

where:
- $W_0$ is **frozen** (pretrained weights)
- $B \in \mathbb{R}^{d \times r}$ and $A \in \mathbb{R}^{r \times k}$ are **trainable**
- $r \ll \min(d, k)$ is the **rank** (typically 1-64)

During forward pass:
$$h = W_0 x + BAx$$

We compute both paths and add them together!

Let's visualize how LoRA decomposes weight updates:

In [ ]:
# Visualize LoRA decomposition
fig, axes = plt.subplots(1, 4, figsize=(16, 3))

d, k, r = 64, 64, 4

# Original weight matrix
W0 = torch.randn(d, k) * 0.1
axes[0].imshow(W0, cmap='RdBu', vmin=-0.3, vmax=0.3)
axes[0].set_title(f'Pretrained W₀\n({d}×{k} = {d*k:,} params)\nFROZEN', fontsize=10, weight='bold')
axes[0].set_xlabel(f'{k} features')
axes[0].set_ylabel(f'{d} features')

# Low-rank matrices
B = torch.randn(d, r) * 0.02
A = torch.randn(r, k) * 0.02

axes[1].imshow(B, cmap='RdBu', vmin=-0.3, vmax=0.3)
axes[1].set_title(f'Matrix B\n({d}×{r} = {d*r} params)\nTRAINABLE', fontsize=10, weight='bold')
axes[1].set_xlabel(f'{r} rank')
axes[1].set_ylabel(f'{d} features')

axes[2].imshow(A, cmap='RdBu', vmin=-0.3, vmax=0.3)
axes[2].set_title(f'Matrix A\n({r}×{k} = {r*k} params)\nTRAINABLE', fontsize=10, weight='bold')
axes[2].set_xlabel(f'{k} features')
axes[2].set_ylabel(f'{r} rank')

# Resulting update
delta_W = B @ A
axes[3].imshow(delta_W, cmap='RdBu', vmin=-0.3, vmax=0.3)
axes[3].set_title(f'Update ΔW = BA\n({d}×{k} = {d*k:,} params)\nLOW RANK', fontsize=10, weight='bold')
axes[3].set_xlabel(f'{k} features')
axes[3].set_ylabel(f'{d} features')

plt.tight_layout()
plt.show()

print(f"\nTrainable parameters: {d*r + r*k:,} instead of {d*k:,}")
print(f"Reduction: {100 * (1 - (d*r + r*k)/(d*k)):.1f}%")

**Key insight**: The update matrix $\Delta W$ has the same shape as $W_0$, but is constructed from much smaller matrices $B$ and $A$.

### 2.3 Rank Choice and Parameter Reduction

The **rank** $r$ is the key hyperparameter in LoRA. Let's see how it affects parameter count:

In [ ]:
# Typical transformer dimensions
d_model = 768  # Hidden dimension (e.g., BERT-base)
ranks = [1, 2, 4, 8, 16, 32, 64]

# Calculate parameters for different ranks
full_params = d_model * d_model
results = []

for r in ranks:
    lora_params = d_model * r + r * d_model
    reduction = (1 - lora_params / full_params) * 100
    results.append({
        'rank': r,
        'params': lora_params,
        'reduction': reduction
    })

print(f"Full attention weight matrix: {d_model}×{d_model} = {full_params:,} parameters")
print("\n" + "="*60)
print(f"{'Rank':<8} {'LoRA Params':<15} {'Reduction':<15} {'% of Original'}")
print("="*60)

for res in results:
    pct = 100 - res['reduction']
    print(f"{res['rank']:<8} {res['params']:<15,} {res['reduction']:<14.2f}% {pct:.3f}%")

**Key observations:**
- Rank 8 (common choice) uses only **2%** of parameters
- Even rank 64 uses only **16.7%** of parameters
- Lower ranks = fewer params but potentially less expressiveness

For a 7B parameter model with LoRA on attention matrices (rank 8), we might only train **~4M parameters** (0.06%)!

## 3. LoRA Architecture

### 3.1 Where to Apply LoRA

In transformer models, we typically apply LoRA to the **attention projection matrices**:
- Query ($W_q$)
- Key ($W_k$) [optional]
- Value ($W_v$)
- Output ($W_o$) [optional]

**Why attention?** These matrices learn task-specific representations. MLP layers can often stay frozen.

Common configurations:
- **LoRA-QV**: Only $W_q$ and $W_v$ (most common)
- **LoRA-QKV**: All three attention matrices
- **LoRA-All**: Attention + MLP layers

Let's visualize a transformer layer with LoRA:

In [ ]:
# Visualize parameter count for different LoRA configurations
d_model = 768
d_ff = 3072  # Feed-forward dimension (typically 4x d_model)
rank = 8

# Calculate parameters
attention_matrices = 4  # Q, K, V, O
mlp_matrices = 2  # Up projection, down projection

configs = {
    'Full Fine-tuning': {
        'attention': attention_matrices * d_model * d_model,
        'mlp': d_model * d_ff + d_ff * d_model,
        'lora': 0
    },
    'LoRA-Q': {
        'attention': attention_matrices * d_model * d_model,
        'mlp': d_model * d_ff + d_ff * d_model,
        'lora': 2 * d_model * rank  # Only Q matrix
    },
    'LoRA-QV': {
        'attention': attention_matrices * d_model * d_model,
        'mlp': d_model * d_ff + d_ff * d_model,
        'lora': 2 * 2 * d_model * rank  # Q and V matrices
    },
    'LoRA-QKV': {
        'attention': attention_matrices * d_model * d_model,
        'mlp': d_model * d_ff + d_ff * d_model,
        'lora': 3 * 2 * d_model * rank  # Q, K, V matrices
    },
    'LoRA-All': {
        'attention': attention_matrices * d_model * d_model,
        'mlp': d_model * d_ff + d_ff * d_model,
        'lora': 4 * 2 * d_model * rank + 2 * (d_model * rank + d_ff * rank)
    }
}

print("Single Transformer Layer Parameter Breakdown:")
print("=" * 80)
print(f"{'Config':<20} {'Total Params':<15} {'Trainable':<15} {'% Trainable'}")
print("=" * 80)

for name, counts in configs.items():
    total = counts['attention'] + counts['mlp']
    trainable = total if name == 'Full Fine-tuning' else counts['lora']
    pct = 100 * trainable / total
    
    print(f"{name:<20} {total:<15,} {trainable:<15,} {pct:.2f}%")

**Key takeaway**: Even LoRA-All trains less than 1% of parameters compared to full fine-tuning!

### 3.2 Initialization Strategy

LoRA uses a clever initialization trick:
- Matrix $A$ is initialized with **random Gaussian** values
- Matrix $B$ is initialized to **zero**

This means $\Delta W = BA = 0$ at initialization, so the model starts with the exact pretrained weights!

We also scale the update by $\frac{\alpha}{r}$ where $\alpha$ is a hyperparameter:
$$h = W_0 x + \frac{\alpha}{r} BAx$$

In [ ]:
# Demonstrate initialization
d, k, r = 512, 512, 8

# Random init for A
A = torch.randn(r, k) / math.sqrt(r)

# Zero init for B
B = torch.zeros(d, r)

# Initial update is zero
delta_W = B @ A

print("Initialization check:")
print(f"A mean: {A.mean():.6f}, std: {A.std():.6f}")
print(f"B mean: {B.mean():.6f}, std: {B.std():.6f}")
print(f"ΔW mean: {delta_W.mean():.6f}, std: {delta_W.std():.6f}")
print(f"\nΔW is zero: {torch.allclose(delta_W, torch.zeros_like(delta_W))}")
print("\nThis ensures the model starts exactly as the pretrained model!")

### 3.3 Merging Adapters for Inference

After training, we can **merge** the LoRA weights into the base model:
$$W' = W_0 + BA$$

This has two benefits:
1. **No inference overhead**: Same speed as the original model
2. **Easy deployment**: Just a single set of weights

The forward pass goes from $W_0 x + BAx$ to $W' x$ with no extra computation!

In [ ]:
# Demonstrate merging
d, k, r = 512, 512, 8
batch_size, seq_len = 32, 128

# Pretrained weight
W0 = torch.randn(d, k) * 0.02

# LoRA weights (after training)
A = torch.randn(r, k) * 0.01
B = torch.randn(d, r) * 0.01

# Input
x = torch.randn(batch_size, seq_len, k)

# Method 1: Separate computation (during training)
output1 = x @ W0.T + x @ (B @ A).T

# Method 2: Merged weights (during inference)
W_merged = W0 + B @ A
output2 = x @ W_merged.T

# Verify they're identical
print("Outputs are identical:", torch.allclose(output1, output2, atol=1e-5))
print(f"\nMax difference: {(output1 - output2).abs().max():.2e}")
print("\nMerging enables zero-overhead inference!")

## 4. Implementation from Scratch

### 4.1 Building the LoRALinear Layer

Let's implement a LoRA-enhanced linear layer from scratch.

In [ ]:
class LoRALinear(nn.Module):
    """
    LoRA-enhanced linear layer.
    
    Replaces W @ x with (W + BA) @ x where:
    - W is frozen pretrained weights
    - B and A are trainable low-rank matrices
    """
    def __init__(
        self,
        in_features: int,
        out_features: int,
        rank: int = 8,
        alpha: float = 16.0,
        bias: bool = True,
        pretrained_weight: Optional[torch.Tensor] = None
    ):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.rank = rank
        self.alpha = alpha
        
        # Pretrained weights (frozen)
        if pretrained_weight is not None:
            self.weight = nn.Parameter(pretrained_weight, requires_grad=False)
        else:
            # Initialize randomly if no pretrained weights
            self.weight = nn.Parameter(torch.randn(out_features, in_features) * 0.02, requires_grad=False)
        
        # Bias (optional, usually frozen)
        if bias:
            self.bias = nn.Parameter(torch.zeros(out_features), requires_grad=False)
        else:
            self.register_parameter('bias', None)
        
        # LoRA low-rank matrices
        self.lora_A = nn.Parameter(torch.randn(rank, in_features) / math.sqrt(rank))
        self.lora_B = nn.Parameter(torch.zeros(out_features, rank))
        
        # Scaling factor
        self.scaling = alpha / rank
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Base transformation: W0 @ x
        result = F.linear(x, self.weight, self.bias)
        
        # LoRA transformation: (α/r) * B @ A @ x
        lora_out = (x @ self.lora_A.T @ self.lora_B.T) * self.scaling
        
        return result + lora_out
    
    def merge_weights(self) -> nn.Linear:
        """Merge LoRA weights into base weights for inference."""
        merged = nn.Linear(self.in_features, self.out_features, bias=self.bias is not None)
        
        # W' = W0 + (α/r) * B @ A
        merged.weight.data = self.weight + (self.lora_B @ self.lora_A) * self.scaling
        
        if self.bias is not None:
            merged.bias.data = self.bias
        
        return merged
    
    def __repr__(self):
        return (f"LoRALinear(in_features={self.in_features}, out_features={self.out_features}, "
                f"rank={self.rank}, alpha={self.alpha})")

print("LoRALinear layer implemented!")

Let's test our LoRALinear layer and verify it works correctly:

In [ ]:
# Create a LoRA linear layer
layer = LoRALinear(in_features=512, out_features=512, rank=8, alpha=16.0)

# Count trainable parameters
total_params = sum(p.numel() for p in layer.parameters())
trainable_params = sum(p.numel() for p in layer.parameters() if p.requires_grad)
frozen_params = total_params - trainable_params

print(f"Total parameters: {total_params:,}")
print(f"Frozen parameters: {frozen_params:,}")
print(f"Trainable parameters (LoRA): {trainable_params:,}")
print(f"\nPercentage trainable: {100 * trainable_params / total_params:.2f}%")

# Test forward pass
x = torch.randn(4, 10, 512)  # (batch, seq_len, features)
output = layer(x)
print(f"\nInput shape: {x.shape}")
print(f"Output shape: {output.shape}")

# Test merging
merged = layer.merge_weights()
output_merged = merged(x)
print(f"\nOutputs match after merging: {torch.allclose(output, output_merged, atol=1e-5)}")

### 4.2 Simple Transformer with LoRA

Now let's build a minimal transformer that uses LoRA in its attention layers.

In [ ]:
class LoRAAttention(nn.Module):
    """
    Multi-head attention with LoRA on Q and V projections.
    """
    def __init__(self, d_model: int, num_heads: int, rank: int = 8, alpha: float = 16.0):
        super().__init__()
        assert d_model % num_heads == 0
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        
        # Query and Value use LoRA (most effective)
        self.query = LoRALinear(d_model, d_model, rank=rank, alpha=alpha)
        self.value = LoRALinear(d_model, d_model, rank=rank, alpha=alpha)
        
        # Key stays frozen (less important for adaptation)
        self.key = nn.Linear(d_model, d_model)
        self.key.weight.requires_grad = False
        self.key.bias.requires_grad = False
        
        # Output projection (frozen)
        self.out = nn.Linear(d_model, d_model)
        self.out.weight.requires_grad = False
        self.out.bias.requires_grad = False
        
    def forward(self, x: torch.Tensor, mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        batch_size, seq_len, d_model = x.shape
        
        # Compute Q, K, V
        Q = self.query(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        K = self.key(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.value(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        
        # Scaled dot-product attention
        scores = (Q @ K.transpose(-2, -1)) / math.sqrt(self.head_dim)
        
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        
        attn_weights = F.softmax(scores, dim=-1)
        attn_output = attn_weights @ V
        
        # Concatenate heads and project
        attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, seq_len, d_model)
        output = self.out(attn_output)
        
        return output

class LoRATransformerBlock(nn.Module):
    """
    Transformer block with LoRA attention and frozen MLP.
    """
    def __init__(self, d_model: int, num_heads: int, d_ff: int, rank: int = 8, alpha: float = 16.0):
        super().__init__()
        
        # Attention with LoRA
        self.attention = LoRAAttention(d_model, num_heads, rank, alpha)
        self.norm1 = nn.LayerNorm(d_model)
        
        # MLP (frozen)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model)
        )
        for param in self.mlp.parameters():
            param.requires_grad = False
        
        self.norm2 = nn.LayerNorm(d_model)
        
    def forward(self, x: torch.Tensor, mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        # Attention block
        x = x + self.attention(self.norm1(x), mask)
        
        # MLP block
        x = x + self.mlp(self.norm2(x))
        
        return x

print("LoRA transformer components implemented!")

Let's create a small transformer and compare parameter counts:

In [ ]:
# Create a LoRA transformer block
d_model, num_heads, d_ff = 256, 4, 1024
rank = 8

block = LoRATransformerBlock(d_model, num_heads, d_ff, rank=rank)

# Count parameters
total_params = sum(p.numel() for p in block.parameters())
trainable_params = sum(p.numel() for p in block.parameters() if p.requires_grad)
frozen_params = total_params - trainable_params

print("LoRA Transformer Block Parameter Breakdown:")
print("=" * 60)
print(f"Total parameters: {total_params:,}")
print(f"Frozen parameters: {frozen_params:,}")
print(f"Trainable parameters (LoRA): {trainable_params:,}")
print(f"\nPercentage trainable: {100 * trainable_params / total_params:.2f}%")

# Test forward pass
x = torch.randn(2, 10, d_model)  # (batch, seq_len, d_model)
output = block(x)

print(f"\nInput shape: {x.shape}")
print(f"Output shape: {output.shape}")

# Breakdown by component
print("\nParameter breakdown:")
for name, module in block.named_modules():
    if isinstance(module, (LoRALinear, nn.Linear)):
        module_params = sum(p.numel() for p in module.parameters())
        module_trainable = sum(p.numel() for p in module.parameters() if p.requires_grad)
        print(f"  {name}: {module_params:,} total, {module_trainable:,} trainable")

### 4.3 Text Classification with LoRA

Let's train a simple sentiment classifier using LoRA. We'll use a toy dataset to demonstrate the concept.

In [ ]:
# Create a simple sentiment dataset
class SentimentDataset(Dataset):
    def __init__(self, size=1000, vocab_size=1000, seq_len=32):
        self.size = size
        self.vocab_size = vocab_size
        self.seq_len = seq_len
        
        # Generate synthetic data
        # Positive: high token values, Negative: low token values
        self.data = []
        for _ in range(size):
            label = torch.randint(0, 2, (1,)).item()
            if label == 1:  # Positive
                tokens = torch.randint(vocab_size // 2, vocab_size, (seq_len,))
            else:  # Negative
                tokens = torch.randint(0, vocab_size // 2, (seq_len,))
            self.data.append((tokens, label))
    
    def __len__(self):
        return self.size
    
    def __getitem__(self, idx):
        return self.data[idx]

# Create datasets
train_dataset = SentimentDataset(size=800)
val_dataset = SentimentDataset(size=200)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)

print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")

# Show example
tokens, label = train_dataset[0]
print(f"\nExample: {tokens[:10]}... → Label: {label}")

Now let's build a simple classifier with LoRA:

In [ ]:
class LoRAClassifier(nn.Module):
    """
    Simple transformer classifier with LoRA.
    """
    def __init__(self, vocab_size: int, d_model: int, num_heads: int, num_layers: int, 
                 num_classes: int, rank: int = 8):
        super().__init__()
        
        # Embeddings (frozen, simulating pretrained)
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.embedding.weight.requires_grad = False
        
        # Transformer layers with LoRA
        self.layers = nn.ModuleList([
            LoRATransformerBlock(d_model, num_heads, d_model * 4, rank=rank)
            for _ in range(num_layers)
        ])
        
        # Classification head (trainable)
        self.classifier = nn.Linear(d_model, num_classes)
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Embed tokens
        x = self.embedding(x)
        
        # Pass through transformer layers
        for layer in self.layers:
            x = layer(x)
        
        # Pool and classify (mean pooling)
        x = x.mean(dim=1)
        logits = self.classifier(x)
        
        return logits

# Create model
model = LoRAClassifier(
    vocab_size=1000,
    d_model=128,
    num_heads=4,
    num_layers=2,
    num_classes=2,
    rank=8
).to(device)

# Count parameters
total = count_parameters(model)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters: {total:,}")
print(f"Trainable parameters: {trainable:,}")
print(f"Percentage trainable: {100 * trainable / total:.2f}%")

Train the model with LoRA:

In [ ]:
# Training setup
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3)
criterion = nn.CrossEntropyLoss()

def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for tokens, labels in loader:
        tokens, labels = tokens.to(device), labels.to(device)
        
        optimizer.zero_grad()
        logits = model(tokens)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        pred = logits.argmax(dim=1)
        correct += (pred == labels).sum().item()
        total += labels.size(0)
    
    return total_loss / len(loader), correct / total

def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for tokens, labels in loader:
            tokens, labels = tokens.to(device), labels.to(device)
            
            logits = model(tokens)
            loss = criterion(logits, labels)
            
            total_loss += loss.item()
            pred = logits.argmax(dim=1)
            correct += (pred == labels).sum().item()
            total += labels.size(0)
    
    return total_loss / len(loader), correct / total

# Train
num_epochs = 10
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

print("Training LoRA model...")
for epoch in range(num_epochs):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, device)
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)
    
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    
    if (epoch + 1) % 2 == 0:
        print(f"Epoch {epoch+1}/{num_epochs}:")
        print(f"  Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
        print(f"  Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

print("\nTraining complete!")

Visualize the training curves:

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss curves
ax1.plot(history['train_loss'], label='Train', marker='o')
ax1.plot(history['val_loss'], label='Validation', marker='s')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training and Validation Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Accuracy curves
ax2.plot(history['train_acc'], label='Train', marker='o')
ax2.plot(history['val_acc'], label='Validation', marker='s')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('Training and Validation Accuracy')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Final validation accuracy: {history['val_acc'][-1]:.4f}")

## 5. QLoRA: Quantization + LoRA

### 5.1 The Memory Problem Persists

Even with LoRA, we still need to:
1. Load the full pretrained model into memory
2. Store activations during forward pass

For a 70B parameter model in FP32:
- Model weights: 70B × 4 bytes = 280 GB
- This exceeds most GPU memory!

**QLoRA** solves this by **quantizing** the base model to 4-bit precision while keeping LoRA adapters in FP16.

### 5.2 Quantization Basics

**Quantization** reduces the precision of weights:
- FP32 (32 bits): Standard floating point
- FP16 (16 bits): Half precision, 2x memory savings
- INT8 (8 bits): Integer, 4x memory savings
- INT4 (4 bits): Aggressive quantization, 8x memory savings

The tradeoff: lower precision = less accuracy (but often minimal for large models).

In [ ]:
# Compare memory requirements with quantization
model_size_b = 70  # 70B parameters
model_size = model_size_b * 1e9

precisions = {
    'FP32': 4,
    'FP16': 2,
    'INT8': 1,
    'INT4 (QLoRA)': 0.5
}

print(f"LLaMA-2 70B Model Memory Requirements:")
print("=" * 60)
print(f"{'Precision':<20} {'Memory (GB)':<15} {'Fits on GPU?'}")
print("=" * 60)

gpu_memory = 80  # A100 80GB

for name, bytes_per_param in precisions.items():
    memory_gb = (model_size * bytes_per_param) / 1e9
    fits = "Yes ✓" if memory_gb < gpu_memory else "No ✗"
    print(f"{name:<20} {memory_gb:<15.1f} {fits}")

print("\nQLoRA makes 70B models trainable on a single A100!")

### 5.3 How QLoRA Works

QLoRA combines three key techniques:

1. **4-bit NormalFloat (NF4)**: Special quantization optimized for normally-distributed weights
2. **Double Quantization**: Quantize the quantization constants to save even more memory
3. **Paged Optimizers**: Use CPU RAM for optimizer states when GPU memory is full

The forward pass:
1. Dequantize 4-bit weights to FP16 (only needed parts, on-the-fly)
2. Compute using FP16
3. Add LoRA updates in FP16

**Key insight**: Frozen weights can be low precision, but trainable weights need higher precision!

Here's a simplified demonstration of quantization:

In [ ]:
def simple_quantize(tensor: torch.Tensor, bits: int = 8) -> Tuple[torch.Tensor, float, float]:
    """
    Simple symmetric quantization.
    
    Maps FP32 values to integer range [-2^(bits-1), 2^(bits-1)-1]
    """
    # Find scale factor
    max_val = tensor.abs().max()
    qmax = 2 ** (bits - 1) - 1
    scale = max_val / qmax
    
    # Quantize
    quantized = torch.round(tensor / scale).clamp(-qmax - 1, qmax)
    
    return quantized, scale, qmax

def dequantize(quantized: torch.Tensor, scale: float) -> torch.Tensor:
    """Dequantize back to FP32."""
    return quantized * scale

# Example: quantize a weight matrix
W = torch.randn(512, 512) * 0.02

# Quantize to 8-bit and 4-bit
W_int8, scale_8, _ = simple_quantize(W, bits=8)
W_int4, scale_4, _ = simple_quantize(W, bits=4)

# Dequantize
W_deq_8 = dequantize(W_int8, scale_8)
W_deq_4 = dequantize(W_int4, scale_4)

# Compute errors
error_8 = (W - W_deq_8).abs().mean()
error_4 = (W - W_deq_4).abs().mean()

print("Quantization Error Analysis:")
print("=" * 60)
print(f"Original: FP32, size = {W.element_size() * W.numel() / 1e6:.2f} MB")
print(f"\nINT8: size = {1 * W.numel() / 1e6:.2f} MB, error = {error_8:.6f}")
print(f"INT4: size = {0.5 * W.numel() / 1e6:.2f} MB, error = {error_4:.6f}")
print(f"\nMemory savings (INT4): {(1 - 0.5 / 4) * 100:.1f}%")

Visualize the effect of quantization:

In [ ]:
# Compare original vs quantized distributions
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Original
axes[0].hist(W.flatten().numpy(), bins=50, alpha=0.7, edgecolor='black')
axes[0].set_title('Original (FP32)')
axes[0].set_xlabel('Value')
axes[0].set_ylabel('Frequency')
axes[0].grid(True, alpha=0.3)

# INT8
axes[1].hist(W_deq_8.flatten().numpy(), bins=50, alpha=0.7, color='orange', edgecolor='black')
axes[1].set_title('Dequantized INT8')
axes[1].set_xlabel('Value')
axes[1].grid(True, alpha=0.3)

# INT4
axes[2].hist(W_deq_4.flatten().numpy(), bins=50, alpha=0.7, color='red', edgecolor='black')
axes[2].set_title('Dequantized INT4 (QLoRA)')
axes[2].set_xlabel('Value')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Notice: INT4 has only 16 unique values, but preserves the overall distribution!")

## 6. Other PEFT Methods

LoRA isn't the only PEFT method. Let's explore alternatives:

### 6.1 Prefix Tuning

**Idea**: Prepend learnable "prefix" tokens to the input sequence.

Instead of modifying weights, we optimize **virtual tokens** $P \in \mathbb{R}^{k \times d}$ (where $k$ is prefix length):
$$\text{Input: } [P_1, P_2, ..., P_k, x_1, x_2, ..., x_n]$$

The model sees these prefixes as context and adapts its behavior accordingly.

In [ ]:
class PrefixTuning(nn.Module):
    """
    Prefix tuning: prepend learnable tokens to the input.
    """
    def __init__(self, prefix_length: int, d_model: int, num_layers: int):
        super().__init__()
        self.prefix_length = prefix_length
        self.d_model = d_model
        
        # Learnable prefix embeddings for each layer
        self.prefix_embeddings = nn.ParameterList([
            nn.Parameter(torch.randn(prefix_length, d_model) * 0.02)
            for _ in range(num_layers)
        ])
    
    def get_prefix(self, layer_idx: int, batch_size: int) -> torch.Tensor:
        """Get prefix for a specific layer and batch size."""
        prefix = self.prefix_embeddings[layer_idx]
        # Expand to batch size: (prefix_len, d_model) -> (batch, prefix_len, d_model)
        return prefix.unsqueeze(0).expand(batch_size, -1, -1)

# Example usage
prefix_tuning = PrefixTuning(prefix_length=10, d_model=256, num_layers=4)

# Count parameters
prefix_params = sum(p.numel() for p in prefix_tuning.parameters())
print(f"Prefix tuning parameters: {prefix_params:,}")
print(f"  = {10} tokens × {256} dim × {4} layers")

# Get prefix for layer 0
batch_size = 4
prefix = prefix_tuning.get_prefix(layer_idx=0, batch_size=batch_size)
print(f"\nPrefix shape: {prefix.shape}  (batch, prefix_len, d_model)")

### 6.2 Prompt Tuning

**Idea**: Similar to prefix tuning, but only optimize the **input embedding layer**.

We learn "soft prompts" - continuous embeddings that act like prompt engineering, but optimized via gradients:
$$\text{Embeddings: } [P_1, P_2, ..., P_k, E(x_1), E(x_2), ..., E(x_n)]$$

Much simpler than prefix tuning (only one layer), but often less powerful.

In [ ]:
class PromptTuning(nn.Module):
    """
    Prompt tuning: learn soft prompt embeddings.
    """
    def __init__(self, num_prompts: int, d_model: int):
        super().__init__()
        self.num_prompts = num_prompts
        
        # Learnable soft prompts
        self.prompts = nn.Parameter(torch.randn(num_prompts, d_model) * 0.02)
    
    def forward(self, input_embeddings: torch.Tensor) -> torch.Tensor:
        """
        Prepend prompts to input embeddings.
        
        Args:
            input_embeddings: (batch, seq_len, d_model)
        
        Returns:
            (batch, num_prompts + seq_len, d_model)
        """
        batch_size = input_embeddings.size(0)
        
        # Expand prompts to batch size
        prompts = self.prompts.unsqueeze(0).expand(batch_size, -1, -1)
        
        # Concatenate prompts with input
        return torch.cat([prompts, input_embeddings], dim=1)

# Example
prompt_tuning = PromptTuning(num_prompts=20, d_model=256)

# Count parameters
prompt_params = sum(p.numel() for p in prompt_tuning.parameters())
print(f"Prompt tuning parameters: {prompt_params:,}")
print(f"  = {20} prompts × {256} dim")

# Apply to input
input_emb = torch.randn(4, 32, 256)  # (batch, seq_len, d_model)
output = prompt_tuning(input_emb)
print(f"\nInput shape: {input_emb.shape}")
print(f"Output shape: {output.shape}  (prompts prepended)")

### 6.3 Adapter Layers

**Idea**: Insert small "adapter" modules between transformer layers.

An adapter is a bottleneck architecture:
$$h' = h + \text{Adapter}(h) = h + W_{\text{up}} \cdot \sigma(W_{\text{down}} \cdot h)$$

where $W_{\text{down}}: d \to r$ (project down) and $W_{\text{up}}: r \to d$ (project up), with $r \ll d$.

In [ ]:
class Adapter(nn.Module):
    """
    Adapter layer with bottleneck architecture.
    """
    def __init__(self, d_model: int, bottleneck_dim: int):
        super().__init__()
        self.down_proj = nn.Linear(d_model, bottleneck_dim)
        self.activation = nn.GELU()
        self.up_proj = nn.Linear(bottleneck_dim, d_model)
        
        # Initialize to near-identity (small changes initially)
        nn.init.zeros_(self.up_proj.weight)
        nn.init.zeros_(self.up_proj.bias)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Residual connection: x + Adapter(x)
        h = self.down_proj(x)
        h = self.activation(h)
        h = self.up_proj(h)
        return x + h

# Example
adapter = Adapter(d_model=256, bottleneck_dim=32)

# Count parameters
adapter_params = sum(p.numel() for p in adapter.parameters())
print(f"Adapter parameters: {adapter_params:,}")
print(f"  Down proj: {256} × {32} = {256 * 32:,}")
print(f"  Up proj: {32} × {256} = {32 * 256:,}")

# Test
x = torch.randn(4, 10, 256)
output = adapter(x)
print(f"\nInput shape: {x.shape}")
print(f"Output shape: {output.shape}  (same as input)")

### 6.4 Comparison of PEFT Methods

Let's compare all PEFT methods side by side:

In [ ]:
# Calculate parameters for each method
d_model = 768
num_layers = 12
num_heads = 12

# Model size (approximate BERT-base)
attn_matrices = 4  # Q, K, V, O
d_ff = d_model * 4
layer_params = attn_matrices * d_model * d_model + 2 * d_model * d_ff
total_params = layer_params * num_layers

methods = {
    'Full Fine-tuning': total_params,
    'LoRA (r=8, QV)': 2 * num_layers * 2 * d_model * 8,
    'LoRA (r=16, QKV)': 3 * num_layers * 2 * d_model * 16,
    'Prefix Tuning (len=10)': 10 * d_model * num_layers,
    'Prompt Tuning (len=20)': 20 * d_model,
    'Adapters (bottleneck=64)': num_layers * 2 * (d_model * 64 + 64 * d_model)
}

print("PEFT Methods Comparison (BERT-base scale):")
print("=" * 70)
print(f"{'Method':<30} {'Parameters':<15} {'% of Total':<15}")
print("=" * 70)

# Sort by parameter count
sorted_methods = sorted(methods.items(), key=lambda x: x[1])

for name, params in sorted_methods:
    pct = 100 * params / total_params
    print(f"{name:<30} {params:>13,}  {pct:>13.3f}%")

print("\nKey insights:")
print("- Prompt tuning: Fewest parameters, simplest, but often weakest")
print("- LoRA: Best performance/efficiency tradeoff, most popular")
print("- Adapters: More parameters than LoRA, but easier to insert/remove")
print("- Prefix tuning: Middle ground between prompt tuning and LoRA")

Visualize the parameter efficiency:

In [ ]:
# Bar chart comparison
fig, ax = plt.subplots(figsize=(12, 6))

names = [name for name, _ in sorted_methods]
params_millions = [params / 1e6 for _, params in sorted_methods]
colors = ['red' if name == 'Full Fine-tuning' else 'steelblue' for name, _ in sorted_methods]

bars = ax.barh(names, params_millions, color=colors, edgecolor='black', alpha=0.7)

# Add value labels
for i, (bar, val) in enumerate(zip(bars, params_millions)):
    pct = 100 * val / (total_params / 1e6)
    label = f"{val:.1f}M ({pct:.2f}%)"
    ax.text(bar.get_width(), bar.get_y() + bar.get_height()/2, 
            label, ha='left', va='center', fontsize=9, weight='bold')

ax.set_xlabel('Trainable Parameters (millions)', fontsize=12, weight='bold')
ax.set_title('PEFT Methods: Parameter Efficiency Comparison', fontsize=14, weight='bold')
ax.set_xlim(0, max(params_millions) * 1.3)
ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

## 7. Practical Guide

### 7.1 When to Use Each Method

Here's a decision guide for choosing a PEFT method:

**Use LoRA when:**
- You want the best performance/efficiency tradeoff
- You're fine-tuning on a specific task (classification, QA, etc.)
- You have some GPU memory (can fit base model + adapters)

**Use QLoRA when:**
- You need to fine-tune very large models (30B+)
- You have limited GPU memory (consumer GPUs)
- Slight accuracy tradeoff is acceptable

**Use Prefix Tuning when:**
- You want to condition model behavior (like prompt engineering)
- You're working with generation tasks
- You want more expressiveness than prompt tuning

**Use Prompt Tuning when:**
- You have very limited compute
- The task is simple or has been seen during pretraining
- You want the absolute minimum parameters

**Use Adapters when:**
- You want to easily swap between tasks
- You're maintaining multiple task-specific models
- You prefer modular architecture

### 7.2 Hyperparameter Guidelines

For **LoRA**, the key hyperparameters are:

1. **Rank (r)**: Start with 8, increase if needed
   - r=4: Very small, fast, but may underfit
   - r=8: Sweet spot for most tasks
   - r=16-32: More capacity, use for complex tasks
   - r=64+: Rarely needed, approaching full fine-tuning cost

2. **Alpha (α)**: Scaling factor, typically 16-32
   - Rule of thumb: α = 2r (e.g., r=8 → α=16)
   - Higher α = stronger LoRA updates
   - Lower α = more conservative, closer to pretrained

3. **Target modules**: Which layers to apply LoRA
   - Start with Q and V (most common)
   - Add K and O if needed
   - Apply to MLP only if attention isn't enough

In [ ]:
# Hyperparameter sensitivity analysis
ranks = [2, 4, 8, 16, 32, 64]
alphas = [8, 16, 32]

d_model = 768
num_layers = 12

print("LoRA Hyperparameter Grid:")
print("=" * 80)
print(f"{'Rank':<8} {'Alpha':<8} {'Params/Layer':<15} {'Total Params (12 layers)'}")
print("=" * 80)

for r in ranks:
    for alpha in alphas:
        params_per_layer = 2 * d_model * r  # For Q and V
        total = params_per_layer * num_layers
        print(f"{r:<8} {alpha:<8} {params_per_layer:<15,} {total:>21,}")
    print("-" * 80)

print("\nRecommendations:")
print("- Start with r=8, α=16")
print("- If underfitting: increase r to 16 or 32")
print("- If overfitting: decrease r to 4 or reduce α")
print("- Higher α makes LoRA updates more aggressive")

### 7.3 Memory Comparison: Full vs LoRA

Let's quantify the memory savings for different model sizes:

In [ ]:
def estimate_lora_memory(model_params_b: float, rank: int = 8, num_layers: int = 32):
    """
    Estimate memory requirements for LoRA fine-tuning.
    
    Assumes:
    - LoRA applied to Q and V in attention (2 matrices per layer)
    - d_model ≈ sqrt(model_params_b * 1e9 / num_layers / 8)
    """
    model_params = model_params_b * 1e9
    
    # Estimate d_model (rough approximation)
    d_model = int((model_params / num_layers / 8) ** 0.5)
    
    # LoRA parameters
    lora_params = 2 * num_layers * 2 * d_model * rank  # 2 matrices * num_layers * 2 dims * rank
    
    # Memory components (GB)
    base_model = model_params * 2 / 1e9  # FP16
    lora_weights = lora_params * 2 / 1e9  # FP16
    gradients = lora_params * 2 / 1e9  # FP16
    optimizer = lora_params * 8 / 1e9  # Adam state
    activations = model_params * 0.1 / 1e9  # Rough estimate
    
    total = base_model + lora_weights + gradients + optimizer + activations
    
    return {
        'base_model_gb': base_model,
        'lora_weights_gb': lora_weights,
        'gradients_gb': gradients,
        'optimizer_gb': optimizer,
        'activations_gb': activations,
        'total_gb': total,
        'lora_params_m': lora_params / 1e6
    }

models = [
    ('GPT-2 Small', 0.124, 12),
    ('GPT-2 Large', 0.774, 36),
    ('LLaMA-2 7B', 7, 32),
    ('LLaMA-2 13B', 13, 40),
    ('LLaMA-2 70B', 70, 80)
]

print("LoRA Memory Requirements (rank=8, FP16):")
print("=" * 90)
print(f"{'Model':<15} {'Base':<8} {'LoRA':<8} {'Grad':<8} {'Opt':<8} {'Act':<8} {'Total':<8} {'LoRA Params'}")
print("=" * 90)

for name, size, layers in models:
    mem = estimate_lora_memory(size, rank=8, num_layers=layers)
    print(f"{name:<15} {mem['base_model_gb']:>6.1f}  {mem['lora_weights_gb']:>6.2f}  "
          f"{mem['gradients_gb']:>6.2f}  {mem['optimizer_gb']:>6.1f}  "
          f"{mem['activations_gb']:>6.1f}  {mem['total_gb']:>6.1f}  {mem['lora_params_m']:>10.1f}M")

print("\nKey insight: Even 70B models can be fine-tuned on a single high-end GPU with LoRA!")

## 8. Production with HuggingFace PEFT

### 8.1 The PEFT Library

While we've implemented LoRA from scratch, in practice you'll use the HuggingFace `peft` library.

**Key advantages:**
- Drop-in compatibility with any HuggingFace model
- Supports LoRA, QLoRA, Prefix Tuning, Prompt Tuning, and more
- Easy adapter saving/loading
- Integration with `transformers` and `accelerate`

Here's how you'd use it in practice (code for reference, not executed):

```python
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from peft import get_peft_model, LoraConfig, TaskType

# Load pretrained model
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

# Configure LoRA
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,  # Rank
    lora_alpha=16,  # Scaling
    lora_dropout=0.1,
    target_modules=["query", "value"],  # Apply to Q and V
    bias="none"
)

# Wrap model with LoRA
model = get_peft_model(model, lora_config)

# Print trainable parameters
model.print_trainable_parameters()
# Output: trainable params: 294,912 || all params: 109,483,778 || trainable%: 0.27%

# Train normally
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset
)
trainer.train()

# Save only the LoRA adapters (tiny!)
model.save_pretrained("./lora_adapters")
# This saves only ~1MB instead of 440MB for full BERT!

# Later: load adapters
from peft import PeftModel
base_model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased")
model = PeftModel.from_pretrained(base_model, "./lora_adapters")
```

### 8.2 QLoRA with BitsAndBytes

For truly large models, you'd use QLoRA with 4-bit quantization:

```python
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model

# Configure 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",  # NormalFloat 4-bit
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True  # Double quantization
)

# Load model in 4-bit
model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Llama-2-7b-hf",
    quantization_config=bnb_config,
    device_map="auto"  # Automatically distribute across GPUs
)

# Prepare for LoRA training
model = prepare_model_for_kbit_training(model)

# Add LoRA adapters
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

# Now you can fine-tune a 7B model on a single 24GB GPU!
```

### 8.3 Multiple Adapters for Multiple Tasks

One powerful PEFT feature: **adapter switching** at inference.

You can train multiple LoRA adapters for different tasks and swap them on the fly:

```python
from peft import PeftModel

# Load base model once
base_model = AutoModelForCausalLM.from_pretrained("gpt2")

# Load different adapters for different tasks
sentiment_model = PeftModel.from_pretrained(base_model, "./adapters/sentiment")
summarization_model = PeftModel.from_pretrained(base_model, "./adapters/summarization")
qa_model = PeftModel.from_pretrained(base_model, "./adapters/qa")

# Or use a single model and swap adapters
model = PeftModel.from_pretrained(base_model, "./adapters/sentiment")

# Inference with sentiment adapter
output = model.generate(input_ids)

# Swap to summarization adapter
model.set_adapter("summarization")
output = model.generate(input_ids)

# Each adapter is only ~1-10MB, so you can store many!
```

This enables a **single base model** serving **multiple tasks** with minimal storage overhead!

## Key Takeaways

### Core Concepts

1. **The Problem**: Full fine-tuning is memory-intensive and can cause catastrophic forgetting

2. **LoRA Solution**: 
   - Freeze pretrained weights
   - Add low-rank decomposition: $W = W_0 + BA$
   - Train only 0.1-1% of parameters
   - Merge adapters for zero-overhead inference

3. **Why Low-Rank Works**: Weight updates during fine-tuning have low intrinsic rank

4. **Key Hyperparameters**:
   - Rank $r$: Start with 8, increase for complex tasks
   - Alpha $\alpha$: Scaling factor, typically $2r$
   - Target modules: Q and V matrices most effective

5. **QLoRA**: Combines 4-bit quantization with LoRA
   - Enables 70B model fine-tuning on consumer GPUs
   - Base model in INT4, adapters in FP16
   - Minimal accuracy loss

6. **Alternative Methods**:
   - Prefix Tuning: Learnable prefix tokens
   - Prompt Tuning: Soft prompts (fewest params)
   - Adapters: Bottleneck layers (easy to swap)

7. **Production**: HuggingFace PEFT library makes this easy
   - Drop-in compatibility
   - Easy adapter saving/loading
   - Multiple adapters per base model

### When to Use

- **LoRA**: Best default choice (performance + efficiency)
- **QLoRA**: Large models (30B+) on limited hardware
- **Prompt Tuning**: Minimal params, simple tasks
- **Adapters**: Multiple tasks, easy swapping

### Memory Savings

| Model | Full Fine-tuning | LoRA (r=8) | QLoRA (4-bit) |
|-------|------------------|------------|---------------|
| GPT-2 (124M) | ~2.2 GB | ~0.5 GB | ~0.3 GB |
| LLaMA-2 7B | ~140 GB | ~25 GB | ~8 GB |
| LLaMA-2 70B | ~1260 GB | ~180 GB | ~45 GB |

**LoRA democratizes LLM fine-tuning!**